# 01. Analyse Exploratoire des Données (EDA) & Propriétés Stylisées

**Projet :** Prévision de l'or (XAU/USD) — *Advanced Time Series* (ENSAI, 3A)

### Objectifs de ce notebook :
1. Charger la série historique du cours de l'or depuis `data/`.
2. Calculer les **log-rendements** : $r_t = \ln(P_t) - \ln(P_{t-1}) = \ln(P_t / P_{t-1})$.
3. Étudier les **faits stylisés** des rendements financiers :
   - Absence d'autocorrélation linéaire dans les rendements (efficience faible des marchés).
   - Distribution non gaussienne : queues épaisses (*fat tails*), asymétrie, excès d'aplatissement (*kurtosis*).
   - Effet ARCH et **clusters de volatilité** : autocorrélation significative des rendements au carré $r_t^2$ ou absolus $|r_t|$.
4. Tester la **stationnarité** des prix et des rendements (tests ADF et KPSS).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Outils économétriques et statistiques
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox
import scipy.stats as stats

# Style des graphiques
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 11

DATA_DIR = Path("../data")
FIG_DIR = Path("../figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)

## 1. Chargement et inspection des données

In [ ]:
data_path = DATA_DIR / "gold_daily.csv"
df = pd.read_csv(data_path, parse_dates=["Date"], index_col="Date")
df = df.sort_index()

print(f"Données chargées : {len(df)} observations du {df.index.min().date()} au {df.index.max().date()}")
df.head()

## 2. Calcul des rendements

On calcule :
- Les **log-rendements journaliers** : $r_t = \ln(P_t / P_{t-1})$
- Les rendements au carré : $r_t^2$ (proxy de volatilité journalière)

In [ ]:
df["log_price"] = np.log(df["Close"])
df["return"] = df["log_price"].diff()
df["squared_return"] = df["return"] ** 2
df["abs_return"] = np.abs(df["return"])

# Retrait de la première valeur NaN
df = df.dropna().copy()
df.head()

## 3. Visualisation : Prix vs Log-rendements

On observe la non-stationnarité en niveau du prix, et l'alternance de périodes calmes et agitées sur les rendements (clusters de volatilité).

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

axes[0].plot(df.index, df["Close"], color="#D4AF37", lw=1.5, label="Cours de l'or (USD)")
axes[0].set_title("Évolution du cours de l'or", fontsize=13, fontweight="bold")
axes[0].set_ylabel("Prix (USD)")
axes[0].legend(loc="upper left")

axes[1].plot(df.index, df["return"], color="#1f77b4", lw=0.8, alpha=0.8, label="Log-rendements")
axes[1].axhline(0, color="black", linestyle="--", lw=0.8)
axes[1].set_title("Log-rendements journaliers (Clusters de volatilité visibles)", fontsize=13, fontweight="bold")
axes[1].set_ylabel("Rendement")
axes[1].legend(loc="upper left")

plt.tight_layout()
plt.savefig(FIG_DIR / "01_price_and_returns.png", dpi=300)
plt.show()

## 4. Statistiques descriptives & Tests de normalité

In [ ]:
ret = df["return"]
stats_summary = {
    "Moyenne (annuelle ~ *252)": ret.mean() * 252,
    "Écart-type (annuel ~ *sqrt(252))": ret.std() * np.sqrt(252),
    "Skewness (Asymétrie)": stats.skew(ret),
    "Kurtosis (Aplatissement total, normal=3)": stats.kurtosis(ret, fisher=False),
    "Excès de Kurtosis": stats.kurtosis(ret, fisher=True)
}

pd.DataFrame(stats_summary, index=["Valeur"]).T

In [ ]:
# Test de Jarque-Bera (H0 : distribution normale)
jb_stat, jb_pvalue = stats.jarque_bera(ret)
print(f"Test de Jarque-Bera : Statistique={jb_stat:.2f}, p-value={jb_pvalue:.4e}")
if jb_pvalue < 0.05:
    print("=> Rejet de la normalité : les rendements ont des queues épaisses (distribution leptokurtique).")

## 5. Tests de stationnarité (ADF & KPSS)

- **Test ADF (Augmented Dickey-Fuller)** : $H_0$ = présence d'une racine unitaire (non stationnaire).
- **Test KPSS** : $H_0$ = série stationnaire.

In [ ]:
def stationarity_summary(series, name="Série"):
    adf_res = adfuller(series.dropna())
    kpss_res = kpss(series.dropna(), regression="c", nlags="auto")
    
    return pd.DataFrame({
        "ADF Stat": [adf_res[0]],
        "ADF p-value": [adf_res[1]],
        "ADF Conclusion (5%)": ["Stationnaire" if adf_res[1] < 0.05 else "Non stationnaire"],
        "KPSS Stat": [kpss_res[0]],
        "KPSS p-value": [kpss_res[1]],
        "KPSS Conclusion (5%)": ["Non stationnaire" if kpss_res[1] < 0.05 else "Stationnaire"]
    }, index=[name])

pd.concat([
    stationarity_summary(df["Close"], name="Prix de clôture"),
    stationarity_summary(df["return"], name="Log-rendements")
])

## 6. Autocorrélations (ACF & PACF) & Effet ARCH

Comparaison de l'autocorrélation :
1. Des rendements $r_t$ (absence de corrélation linéaire => imprévisibilité linéaire).
2. Des rendements au carré $r_t^2$ (forte corrélation => dépendance non linéaire / effet ARCH).

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

plot_acf(df["return"], lags=40, ax=axes[0, 0], title="ACF - Log-rendements (r_t)")
plot_pacf(df["return"], lags=40, ax=axes[0, 1], title="PACF - Log-rendements (r_t)", method="ywm")

plot_acf(df["squared_return"], lags=40, ax=axes[1, 0], title="ACF - Rendements au carré (r_t^2) [Effet ARCH]")
plot_pacf(df["squared_return"], lags=40, ax=axes[1, 1], title="PACF - Rendements au carré (r_t^2)", method="ywm")

plt.tight_layout()
plt.savefig(FIG_DIR / "01_acf_pacf.png", dpi=300)
plt.show()